# Week 2: hyperparameter grid search
Authors: CS163 RFI team; grid analysis and imbalance experiments: Selina.

Run six independent trials for one architecture/width pair. Run all four pairs (`cnn`, `unet` × 8, 16) for the full 24-setting grid. Select on validation only. See the folder README for the protocol and reference results.


## 1. Choose architecture and width
All four pairs (`cnn`, `unet` × 8, 16) comprise 24 independent trials. Override `RFI_ARCHITECTURE` and `RFI_BASE_CHANNELS` before launching.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
ARCHITECTURE = os.environ.get("RFI_ARCHITECTURE", "cnn")  # cnn or unet
from cs163_rfi import paths  # folders come from .env at the repo root
OWNER = paths.NAME  # RFI_NAME in .env
PROJECT_ROOT = paths.REPO_ROOT
DATA = paths.DATA_DIR
BASE_CHANNELS = int(os.environ.get("RFI_BASE_CHANNELS", 8))  # widths: 8→16→32 or 16→32→64
LOSSES = ("bce", "mse", "l1")  # MSE is squared L2; do not add duplicate L2.
LEARNING_RATES = (3e-4, 1e-3)
SEED = 163
BATCH_SIZE = 1
PROBABILITY_BINS = 65536
MIN_EPOCHS = 20
PATIENCE = 10
MIN_AP_IMPROVEMENT = 0.001
MAX_EPOCHS = 100  # Initial cap per trial; increase if the grid pauses before a plateau.
GRID_RESUME_DIR = os.environ.get("RFI_GRID_RESUME_DIR", "")  # To resume, paste the printed GRID DIRECTORY here.
# In the same runtime, rerunning Run all reuses its current grid automatically.
# To deliberately start a NEW grid, clear GRID_DIRECTORY from memory first.

## 2. Prepare output folders


In [ ]:
assert DATA.is_dir(), f"Dataset not found: {DATA}. Configure .env first."
SAVE_ROOT = paths.RESULTS_DIR / "runs"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Data:", DATA)
print("Results:", SAVE_ROOT)
print("Checkpoints:", paths.CHECKPOINT_DIR)
LOCAL_ARCHIVE = None  # Original data checks run without requiring a local starter ZIP.


In [ ]:
from cs163_rfi.experiment import (
    inspect_dataset, fit_training_normalization,
    initialize_grid, write_json, run_six_trial_grid,
    sha256_file, WORKFLOW_VERSION, BASELINE_FAR,
)
import cs163_rfi.experiment as workflow
import torch
from IPython.display import display


## 3. Verify data and train-only normalization
One mean/std pools all five clean training recordings across all antennas, times and frequencies. Labels and classical-detector inputs stay unchanged.


In [ ]:
ROOT = PROJECT_ROOT
assert (ROOT / "pyproject.toml").is_file()
DATA, catalog, DATA_REPORT = inspect_dataset(ROOT, DATA, LOCAL_ARCHIVE)
NORM_MEAN, NORM_STD = fit_training_normalization(DATA, catalog)
NORMALIZATION = {"mean": NORM_MEAN, "std": NORM_STD}
display(catalog.groupby("split").size().rename("recordings").to_frame())
print("Normalization:", NORMALIZATION)
print("Data checks:", DATA_REPORT)

## 4. Run or resume the six-trial grid
The shared module contains dataset checks, models, losses, training/resume, calibration and evaluation, in that order. Checkpoints are selected by validation AP; thresholds match the SK+bandpass clean-validation FPR. Training uses at least 20 epochs, patience 10 and improvement anchor 0.001. A budget-exhausted trial remains unfinished.


In [ ]:
assert ARCHITECTURE in ("cnn", "unet")
assert OWNER.strip() and OWNER != "YOUR_NAME", "Set OWNER before starting."
assert BASE_CHANNELS in (8, 16) and BATCH_SIZE == 1
assert tuple(LOSSES) == ("bce", "mse", "l1") and tuple(LEARNING_RATES) == (3e-4, 1e-3)
assert MIN_EPOCHS >= 1 and PATIENCE >= 1 and MAX_EPOCHS >= MIN_EPOCHS
PROTOCOL = {"workflow_version": WORKFLOW_VERSION, "architecture": ARCHITECTURE,
    "base_channels": BASE_CHANNELS, "seed": SEED, "batch_size": BATCH_SIZE,
    "probability_bins": PROBABILITY_BINS, "min_epochs": MIN_EPOCHS, "patience": PATIENCE,
    "min_ap_improvement": MIN_AP_IMPROVEMENT, "archive_sha256": DATA_REPORT["archive_sha256"],
    "catalog_sha256": DATA_REPORT["catalog_sha256"], "baseline_far": BASELINE_FAR,
    "shared_code_sha256": sha256_file(workflow.__file__)}
resume_path = GRID_RESUME_DIR or globals().get("GRID_DIRECTORY", "")
GRID_DIRECTORY, GRID_MANIFEST = initialize_grid(SAVE_ROOT, OWNER, PROTOCOL, LOSSES, LEARNING_RATES, resume_path)
write_json(GRID_DIRECTORY / "data_checks.json", DATA_REPORT)
print("GRID DIRECTORY — paste into GRID_RESUME_DIR after a reset:", GRID_DIRECTORY)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu": print("CPU works but the full grid is slow; request a GPU in the Slurm job.")
GRID_SUMMARY = run_six_trial_grid(GRID_DIRECTORY, GRID_MANIFEST, DATA, catalog, NORMALIZATION, device, MAX_EPOCHS)
display(GRID_SUMMARY)
print("Saved grid summary:", GRID_DIRECTORY / "grid_summary.csv")
print("Completed trials:", int(GRID_SUMMARY.status.eq("complete").sum()), "/ 6")

In [ ]:
print("Saved:", GRID_DIRECTORY)
